## Long-Term Terminus Change Prediction - with XGBoost

In [ ]:
# dependencies

import os
import json
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import ipynbname
from sklearn.metrics import mean_squared_error, r2_score, precision_score, recall_score
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler
from scipy.signal import argrelextrema
from scipy import stats
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll.base import scope
import copy
from PIL import Image, ImageChops
import joblib

import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", message=".*The 'nopython' keyword.*")  # from shap

import shap

In [ ]:
# objective function for hyperparameters
def objective(params):
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    score = -r2_score(y_test, y_pred)  # Negative R2 score for minimization
    return {'loss': score, 'status': STATUS_OK}

In [ ]:
filepath = "/path/to/your/dataframe_csv/files/here/"
outdir = "/your_output_path_here/outputs/prediction_plots/"

In [ ]:
# import dataframes
filepath = filepath
data_dict = {}

for file in os.listdir(filepath):
    if file.endswith(".csv"):
        glacier_id = file[:3]  
        df = pd.read_csv(os.path.join(filepath, file), header=0)
        df["GID"] = glacier_id 
        cols = ["GID"] + [col for col in df.columns if col != "GID"]
        df = df[cols]
        data_dict[f"data_{glacier_id}"] = df

data = pd.concat(data_dict.values(), ignore_index=True)
data = data.sort_values(by=["GID", "Date"]).reset_index(drop=True)

In [ ]:
# Set all "Longterm ..." columns to start at 0
def normalize_longterm(group):
    # Find all longterm columns
    longterm_cols = [col for col in group.columns if col.startswith("Longterm ")]
    
    for col in longterm_cols:
        # Find the first non-NaN value
        first_valid = group[col].dropna().iloc[0]
        # Subtract the first value from all values
        group[col] = group[col] - first_valid
    
    return group

# Apply normalization per GID
data = data.groupby("GID", group_keys=False).apply(normalize_longterm)

# Rename columns
rename_map = {
    col: f"Change in {col.replace('Longterm ', '')}"
    for col in data.columns if col.startswith("Longterm ")
}
data = data.rename(columns=rename_map)

In [ ]:
# --- Define target + inputs ---
target_col = 'Change in Terminus Position'
input_cols = [col for col in data.columns if col.startswith("Change in ") and col != target_col]
retain_cols = ['GID', 'Date']

X_ref = data[retain_cols + input_cols]
y_ref = data[retain_cols + [target_col]]
X = data[input_cols]
y = data[target_col]

# --- Output directory ---
outdir = outdir
os.makedirs(outdir, exist_ok=True)

In [ ]:
# chosen GIDs for training, or GIDs can be randomly selected
train_unique_gids = [
    '001', '002', '004', '011', '016', '030', '031', '035', 
    '051', '052', '056', '061', '065', '077', '080', '113', '116', '121', 
    '171', '181', '189', '223', '229', '240', '247', '248', 
    '273', '275', '279', '280', '288'
    ]

### Model training run

In [ ]:
test_gid = "290"   # GID: ideally a glacier similar to your desired behavior, but not in the training set

print(f"\n➡️ Running single-glacier model for GID {test_gid}...")

# Train/test split
train_gids = [gid for gid in train_unique_gids if gid != test_gid]
test_gids = [test_gid]

train_mask = data['GID'].isin(train_gids)
test_mask = data['GID'].isin(test_gids)

X_train = X[train_mask].reset_index(drop=True)
X_test = X[test_mask].reset_index(drop=True)
y_train = y[train_mask].reset_index(drop=True)
y_test = y[test_mask].reset_index(drop=True)

X_train_ref = X_ref[train_mask].reset_index(drop=True)
X_test_ref = X_ref[test_mask].reset_index(drop=True)
y_train_ref = y_ref[train_mask].reset_index(drop=True)
y_test_ref = y_ref[test_mask].reset_index(drop=True)

# --- Hyperopt parameter search ---

param_space = {
    'tree_method': hp.choice('tree_method', ['approx']),  
    'objective': hp.choice('objective', ['reg:squarederror']),  
    'n_estimators': scope.int(hp.quniform('n_estimators', 100, 2000, 50)),
    'max_depth': scope.int(hp.randint('max_depth', 2, 15)),
    'learning_rate': hp.uniform('learning_rate', 0.001, 0.2),
    'alpha': hp.uniform('alpha', 0, 100), 
    'reg_lambda': hp.uniform('reg_lambda', 1, 20), 
    'max_delta_step': scope.int(hp.randint('max_delta_step', 0, 50)),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.2, 1.0),
    'subsample': hp.uniform('subsample', 0.5, 1.0),
    'colsample_bylevel': hp.uniform('colsample_bylevel', 0.5, 1.0),
    'min_child_weight': scope.int(hp.quniform('min_child_weight', 1, 10, 1))
}

trials = Trials()
best = fmin(
    fn=objective,
    space=param_space,
    algo=tpe.suggest,
    max_evals=100,
    trials=trials,
    rstate=np.random.default_rng(42)
)

best_params = {
    'tree_method': 'approx',
    'objective': 'reg:squarederror',
    'n_estimators': int(best['n_estimators']),
    'max_depth': int(best['max_depth']),
    'learning_rate': best['learning_rate'],
    'alpha': best['alpha'],
    'reg_lambda': best['reg_lambda'],
    'max_delta_step': best['max_delta_step'],
    'colsample_bytree': best['colsample_bytree'],
    'subsample': best['subsample'],
    'colsample_bylevel': best['colsample_bylevel'],
    'min_child_weight': best['min_child_weight']
}

# --- Train model ---
print("   ✅ Training model...")
evalset = [(X_train, y_train), (X_test, y_test)]
model = xgb.XGBRegressor(**best_params, random_state=42)
model.fit(X_train, y_train, eval_set=evalset, verbose=False)

# --- Save model for reuse ---
model_outfile = os.path.join(
    "/directory/where/you/want/your/model/saved",
    f"long_term_model.joblib"
)
joblib.dump(model, model_outfile)
print(f"   💾 Model saved to {model_outfile}")

# --- Evaluate predictions (optional quick check) ---
pred_test = model.predict(X_test)
test_rmse = mean_absolute_error(y_test, pred_test)
test_r2 = r2_score(y_test, pred_test)
print(f"   📊 RMSE={test_rmse:.2f}, R²={test_r2:.2f}")


In [ ]:
# --- SHAP Time-Series SHAP Storage for training model --- #

force_out_dir = outdir + "SHAP_force_and_data"
os.makedirs(force_out_dir, exist_ok=True)

print("   🔍 Running SHAP analysis for FULL training dataset...")

# Compute SHAP values
explainer = shap.Explainer(model, X_train)
shap_train = explainer(X_train)

# Create MultiIndex: (GID, Date) for each training row
train_gids_series = data.loc[train_mask, 'GID'].reset_index(drop=True)
train_dates_series = X_train_ref['Date'].astype('datetime64[ns]').reset_index(drop=True)

multi_index = pd.MultiIndex.from_arrays(
    [train_gids_series, train_dates_series],
    names=['GID', 'Date']
)

# Construct dataframe of SHAP values
shap_train_df = pd.DataFrame(
    shap_train.values,
    index=multi_index,
    columns=X_train.columns
)

shap_train_df = shap_train_df.sort_index(level=['GID','Date'])

# Save to CSV
shap_data_file = os.path.join(force_out_dir, f"train_SHAP_full_timeseries.csv")
shap_train_df.to_csv(shap_data_file)

print(f"   💾 SHAP training time-series data saved to:\n       {shap_data_file}")

### Use trained model on unused glacier data

In [ ]:
exclusin_list = []

# # uncomment and use if you need to exclude any glaciers you have data for

# exclusion_list = [
#             '007','021','044','090','088','099','132','195','212','227','252','278','291'
#                  ]

In [ ]:
# --- Reload the saved model ---
model_path = os.path.join(
    "/directory/where/you/want/your/model/saved",
    f"long_term_model_v6.joblib"
)
model = joblib.load(model_path)
print(f"✅ Loaded model from {model_path}")

# --- Exclude glaciers that were in training or on the exclusion list ---
excluded_gids = train_unique_gids  # glaciers used in training
mask_new = ~data['GID'].isin(excluded_gids) & ~data['GID'].isin(exclusion_list)

X_new_ref = X_ref[mask_new].reset_index(drop=True)
X_new = X_new_ref.drop(columns=['GID', 'Date'])

# --- Run predictions on new glacier systems ---
pred_new = model.predict(X_new)

# --- Attach predictions back to dataframe for inspection ---
results_new = X_new_ref.copy()
results_new['Predicted Change in Terminus Position'] = pred_new

# --- Output directory ---
outdir = outdir + "additional_all"
os.makedirs(outdir, exist_ok=True)

summary_metrics_all = []

In [ ]:
# --- Loop through each new glacier and generate plots + metrics ---
for gid in sorted(results_new["GID"].unique()):
    gid_str = str(gid).zfill(3)

    # Subset data for this glacier
    X_ref_gid = X_ref[X_ref["GID"] == gid]
    y_ref_gid = y_ref[y_ref["GID"] == gid][target_col]
    pred_gid = results_new[results_new["GID"] == gid]["Predicted Change in Terminus Position"]

    # Shift long-term trend + prediction to start at 0
    if len(y_ref_gid) > 0:
        y_longterm_shifted = y_ref_gid.values.ravel() - y_ref_gid.values.ravel()[0]
        pred_shifted = pred_gid.values.ravel() - pred_gid.values.ravel()[0]
    else:
        y_longterm_shifted = y_ref_gid.values.ravel()
        pred_shifted = pred_gid.values.ravel()

    # Load original terminus data
    orig_file = f"/path/to/your/un_interpolated/terminus/position/data.csv"
    if os.path.exists(orig_file):
        orig_df = pd.read_csv(orig_file)
        if "Date" in orig_df.columns and "Terminus Change" in orig_df.columns:
            orig_df["Date"] = pd.to_datetime(orig_df["Date"], errors="coerce")
        else:
            orig_df = pd.DataFrame(columns=["Date", "Terminus Change"])
    else:
        orig_df = pd.DataFrame(columns=["Date", "Terminus Change"])
    
    # Reference dates and series
    x_dates = X_ref_gid["Date"].values.astype("datetime64[ns]")
    y_actual = y_longterm_shifted
    y_pred = pred_shifted
    date_min, date_max = x_dates.min(), x_dates.max()
    
    # Filter original data to the same date range
    orig_plot_df = orig_df[(orig_df["Date"] >= date_min) & (orig_df["Date"] <= date_max)].copy()
    
    # Shift original data so the first plotted point is 0
    if not orig_plot_df["Terminus Change"].dropna().empty:
        first_val = orig_plot_df["Terminus Change"].dropna().iloc[0]
        orig_plot_df["Terminus Change"] = orig_plot_df["Terminus Change"] - first_val
    
    x_dates_orig = orig_plot_df["Date"].values.astype("datetime64[ns]")
    y_orig = orig_plot_df["Terminus Change"].values
    
    # Metrics
    rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
    y_range = np.max(y_actual) - np.min(y_actual)
    nrmse = rmse / y_range if y_range != 0 else np.nan
    r2 = r2_score(y_actual, y_pred)
    spearman_stat, spearman_p = stats.spearmanr(y_actual, y_pred)
    
    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(x_dates, y_actual, label="Long-Term Trend", marker="o", markersize=4)
    plt.plot(x_dates, y_pred, label="Model Prediction", marker="x", markersize=4)
    plt.scatter(x_dates_orig, y_orig, label="Original Data", s=12, marker=".", alpha=0.8, color="gray")
    plt.title(f"Glacier {gid_str} — Predictions\nR²={r2:.2f}, RMSE={rmse:.2f}")
    plt.xlabel("Year")
    plt.ylabel("Change in Terminus Position")
    plt.legend()
    plt.tight_layout()
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.xticks(rotation=45)
    
    # Save figure
    outfile = os.path.join(outdir, f"GID_{gid_str}_predictions.png")
    plt.savefig(outfile, dpi=300)
    plt.close()


    # Save metrics
    summary_metrics_all.append({
        "GID": gid_str,
        "RMSE": rmse,
        "NRMSE": nrmse,
        "R2": r2,
        "Spearman_Statistic": spearman_stat,
        "Spearman_PValue": spearman_p
    })

    # ========================
    # Save plotted data as CSV
    # ========================
    pred_data_dir = outdir
    os.makedirs(pred_data_dir, exist_ok=True)

    # Create dataframe with Long-Term Trend and Model Prediction
    plot_df = pd.DataFrame({
        "Date": x_dates,
        "Long-Term Trend": y_actual,
        "Model Prediction": y_pred
    })

    # Add Original Data as separate columns with its own dates
    if len(orig_plot_df) > 0:
        plot_df["Date Original"] = pd.Series(x_dates_orig)
        plot_df["Original Data"] = pd.Series(y_orig)
    else:
        plot_df["Date Original"] = pd.NaT
        plot_df["Original Data"] = np.nan

    # Save to CSV
    data_outfile = os.path.join(pred_data_dir, f"GID_{gid_str}_prediction_data.csv")
    plot_df.to_csv(data_outfile, index=False)


# Save metrics summary
metrics_df = pd.DataFrame(summary_metrics_all)
metrics_outfile = os.path.join(outdir, f"summary_metrics_new_predictions.csv")
metrics_df.to_csv(metrics_outfile, index=False)
print(f"✅ Metrics summary saved to {metrics_outfile}")

### Training cycle loop - run this to cycle out one glacier from the training set at a time so we can get prediction results for it.

In [ ]:
# same as before + the chosen test_gid
train_unique_gids = ['001', '002', '004', '011', '016', '030', '031', '035', 
                     '051', '052', '056', '061', '065', '077', '080', '113', '116', '121', 
                     '171', '181', '189', '223', '229', '240', '247', '248', 
                     '273', '275', '279', '280', '288', '290'
                     ]

In [ ]:
train_outdir = "/path/to/your/outputs/prediction_plots/training_all"
os.makedirs(train_outdir, exist_ok=True)

# Directory for saving training data CSVs
train_data_outdir = "/path/to/your/outputs/prediction_data/training_all"
os.makedirs(train_data_outdir, exist_ok=True)

summary_metrics_all = []
unique_gids = train_unique_gids

for test_gid in unique_gids:
    print(f"\n➡️ Processing Test GID {test_gid}...")

    # Train/test split
    train_gids = [gid for gid in unique_gids if gid != test_gid]
    test_gids = [test_gid]

    train_mask = data['GID'].isin(train_gids)
    test_mask = data['GID'].isin(test_gids)

    X_train = X[train_mask].reset_index(drop=True)
    X_test = X[test_mask].reset_index(drop=True)
    y_train = y[train_mask].reset_index(drop=True)
    y_test = y[test_mask].reset_index(drop=True)

    X_train_ref = X_ref[train_mask].reset_index(drop=True)
    X_test_ref = X_ref[test_mask].reset_index(drop=True)
    y_train_ref = y_ref[train_mask].reset_index(drop=True)
    y_test_ref = y_ref[test_mask].reset_index(drop=True)

    # --- Hyperopt parameter search ---
    param_space = {
        'tree_method': hp.choice('tree_method', ['approx']),  
        'objective': hp.choice('objective', ['reg:squarederror']),  
        'n_estimators': scope.int(hp.quniform('n_estimators', 100, 2000, 50)),
        'max_depth': scope.int(hp.randint('max_depth', 2, 15)),
        'learning_rate': hp.uniform('learning_rate', 0.001, 0.2),
        'alpha': hp.uniform('alpha', 0, 100), 
        'reg_lambda': hp.uniform('reg_lambda', 1, 20), 
        'max_delta_step': scope.int(hp.randint('max_delta_step', 0, 50)),
        'colsample_bytree': hp.uniform('colsample_bytree', 0.2, 1.0),
        'subsample': hp.uniform('subsample', 0.5, 1.0),
        'colsample_bylevel': hp.uniform('colsample_bylevel', 0.5, 1.0),
        'min_child_weight': scope.int(hp.quniform('min_child_weight', 1, 10, 1))
    }

    trials = Trials()
    
    # Replace with your objective function
    best = fmin(fn=objective,
                space=param_space,
                algo=tpe.suggest,
                max_evals=100,
                trials=trials,
                rstate=np.random.default_rng(42))

    best_params = {
        'tree_method': 'approx',
        'objective': 'reg:squarederror',
        'n_estimators': int(best['n_estimators']),
        'max_depth': int(best['max_depth']),
        'learning_rate': best['learning_rate'],
        'alpha': best['alpha'],
        'reg_lambda': best['reg_lambda'],
        'max_delta_step': best['max_delta_step'],
        'colsample_bytree': best['colsample_bytree'],
        'subsample': best['subsample'],
        'colsample_bylevel': best['colsample_bylevel'],
        'min_child_weight': best['min_child_weight']
    }

    # --- Loop over test glaciers (here only one) ---
    for gid in test_gids:
        print(f"   ✅ Running model for GID {gid}")

        X_test_ref_gid = X_test_ref[X_test_ref['GID'] == gid]
        y_test_ref_gid = y_test_ref[y_test_ref['GID'] == gid]

        X_test_gid = X_test_ref_gid.drop(columns=['GID', 'Date'])
        y_test_gid = y_test_ref_gid.drop(columns=['GID', 'Date'])

        # Fit model
        evalset = [(X_train, y_train), (X_test_gid, y_test_gid)]
        model = xgb.XGBRegressor(**best_params, random_state=42)
        model.fit(X_train, y_train, eval_set=evalset, verbose=False)

        # Predictions
        pred_test = model.predict(X_test_gid)

        # --- Shift all series to start at zero BEFORE metrics ---
        if len(y_test_gid) > 0:
            y_test_gid_shifted = y_test_gid.values.ravel() - y_test_gid.values.ravel()[0]
            pred_test_shifted = pred_test.ravel() - pred_test.ravel()[0]
        else:
            y_test_gid_shifted = y_test_gid.values.ravel()
            pred_test_shifted = pred_test.ravel()

        # Original data for plotting
        orig_file = f"/path/to/your/un_interpolated/terminus/position/data_{str(gid).zfill(3)}.csv"
        if os.path.exists(orig_file):
            orig_df = pd.read_csv(orig_file)
            if "Date" in orig_df.columns and "Terminus Change" in orig_df.columns:
                orig_df["Date"] = pd.to_datetime(orig_df["Date"], errors="coerce")

                # Shift original series to start at 0
                if not orig_df["Terminus Change"].dropna().empty:
                    orig_df["Terminus Change"] = orig_df["Terminus Change"] - orig_df["Terminus Change"].iloc[0]
            else:
                orig_df = pd.DataFrame(columns=["Date", "Terminus Change"])
        else:
            orig_df = pd.DataFrame(columns=["Date", "Terminus Change"])


        # Dates
        x_dates_test = X_test_ref_gid['Date'].values.astype('datetime64[ns]')
        y_actual = y_test_gid_shifted
        y_pred = pred_test_shifted

        date_min, date_max = x_dates_test.min(), x_dates_test.max()
        orig_plot_df = orig_df[(orig_df["Date"] >= date_min) & (orig_df["Date"] <= date_max)]
        x_dates_orig = orig_plot_df["Date"].values.astype('datetime64[ns]')
        y_orig = orig_plot_df["Terminus Change"].values

        # Metrics
        rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
        y_range = np.max(y_actual) - np.min(y_actual)
        nrmse = rmse / y_range if y_range != 0 else np.nan
        r2 = r2_score(y_actual, y_pred)
        spearman_stat, spearman_p = stats.spearmanr(y_actual, y_pred)

        # Plot
        plt.figure(figsize=(10, 5))
        plt.plot(x_dates_test, y_actual, label="Long-Term Trend", marker="o", markersize=4)
        plt.plot(x_dates_test, y_pred, label="Predicted", marker="x", markersize=4)
        plt.scatter(x_dates_orig, y_orig, label="Original Data", s=12, marker=".", alpha=0.8, color="gray")
        plt.title(f"Glacier {gid} — Test Predictions\nR²={test_r2:.2f}, RMSE={test_rmse:.2f}")
        plt.xlabel("Year")
        plt.ylabel("Value (relative to start)")
        plt.legend()
        plt.tight_layout()
        plt.gca().xaxis.set_major_locator(mdates.YearLocator())
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        plt.xticks(rotation=45)
        plt.savefig(os.path.join(train_outdir, f"GID_{gid}_test_predictions.png"), dpi=300)
        plt.close()

        # ================================
        # Save plotted data to CSV
        # ================================
        train_data_file = os.path.join(train_data_outdir, f"GID_{gid}_training_data.csv")

        plot_df = pd.DataFrame({
            "Date": x_dates_test,
            "Long-Term Trend": y_actual,
            "Model Prediction": y_pred
        })

        if len(orig_plot_df) > 0:
            plot_df["Date Original"] = pd.Series(x_dates_orig)
            plot_df["Original Data"] = pd.Series(y_orig)
        else:
            plot_df["Date Original"] = pd.NaT
            plot_df["Original Data"] = np.nan

        plot_df.to_csv(train_data_file, index=False)

        
        # Save metrics
        summary_metrics_all.append({
            "GID": gid,
            "RMSE": rmse,
            "NRMSE": nrmse,
            "R2": r2,
            "Spearman_Statistic": spearman_stat,
            "Spearman_PValue": spearman_p
        })

# Save summary metrics
summary_df = pd.DataFrame(summary_metrics_all)
summary_df.to_csv(os.path.join(train_outdir, "all_glaciers_test_metrics_summary.csv"), index=False)
print("✅ All leave-one-glacier-out predictions and summary metrics saved.")